# 🛡️ ĐỒ ÁN KHAI PHÁ DỮ LIỆU: HỆ THỐNG PHÁT HIỆN XÂM NHẬP MẠNG LAI (HYBRID ML-NIDS)
### Mô hình kết hợp: Học không giám sát (Mahalanobis / K-Means / DBSCAN) & Học có giám sát (Random Forest) & Tích hợp SIEM Wazuh
---
**Mục tiêu của Notebook này:**
1. **Tiền xử lý toàn diện:** Nén độ lệch phải bằng biến đổi $\text{Log1p}$, mã hóa danh định One-Hot Encoding và chuẩn hóa $Z$-Score.
2. **Động cơ học không giám sát:** Giảm chiều PCA 8 thành phần, phân cụm Mini-Batch K-Means tạo hồ sơ lưu lượng chuẩn (Normal Baseline), phân lập nhiễu DBSCAN và tính khoảng cách elip Mahalanobis với ngưỡng bách phân vị thứ 95 để tóm gọn các cuộc tấn công chưa từng biết (Zero-Day).
3. **Mô hình học có giám sát:** Rừng cây quyết định (Random Forest 5 lớp) định danh chính xác các dòng tấn công đã biết (DoS, Probe, R2L, U2R).
4. **Thực nghiệm đối đầu & SIEM:** Chứng minh tính ưu việt khi bắt Zero-Day và tự động gửi cảnh báo Syslog RFC 5424 về Wazuh Manager.


## 📌 BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG & CẤU HÌNH ĐẶC TRƯNG MẠNG
- Khóa số luồng OpenBLAS / OMP về 1 để tối ưu bộ nhớ và tránh lỗi xung đột tiến trình trên Windows / Colab.
- Định nghĩa 41 thuộc tính của bộ dữ liệu chuẩn an ninh mạng **NSL-KDD**.
- Ánh xạ các kỹ thuật tấn công sang ma trận **MITRE ATT&CK** và mã định danh Rule Wazuh.


In [ ]:
import os
# Khóa số luồng để đảm bảo an toàn bộ nhớ
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import gc
import json
import socket
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Thiết lập đường dẫn dữ liệu tự động
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data"
TRAIN_FILE = DATA_DIR / "KDDTrain+.txt"
if not TRAIN_FILE.exists():
    # Dự phòng đường dẫn trên máy local nếu chưa có trong data/
    alt_file = Path(r"D:\data_miner\archive\KDDTrain+.txt")
    if alt_file.exists():
        TRAIN_FILE = alt_file

print(f"✅ Thư viện đã nạp xong! Đường dẫn dữ liệu huấn luyện: {TRAIN_FILE}")

# Danh sách 41 đặc trưng của tập NSL-KDD
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']
SKEWED_COLS = ['duration', 'src_bytes', 'dst_bytes']
NUMERIC_COLS = [c for c in COLUMN_NAMES if c not in CATEGORICAL_COLS + ['label', 'difficulty']]

# Bảng ánh xạ 5 nhóm tấn công chính
ATTACK_MAP = {
    'normal': 'Normal',
    'neptune': 'DoS', 'smurf': 'DoS', 'pod': 'DoS', 'teardrop': 'DoS', 'land': 'DoS', 'back': 'DoS',
    'apache2': 'DoS', 'udpstorm': 'DoS', 'processtable': 'DoS', 'mailbomb': 'DoS',
    'portsweep': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe', 'satan': 'Probe', 'saint': 'Probe', 'mscan': 'Probe',
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'phf': 'R2L', 'multihop': 'R2L',
    'warezmaster': 'R2L', 'warezclient': 'R2L', 'spy': 'R2L', 'snmpguess': 'R2L', 'httptunnel': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R', 'ps': 'U2R', 'xterm': 'U2R'
}

MITRE_INFO = {
    'Normal': {'id': 'N/A', 'name': 'Normal Traffic', 'rule_id': 100199, 'level': 3},
    'DoS':    {'id': 'T1499', 'name': 'Endpoint Denial of Service', 'rule_id': 100202, 'level': 10},
    'Probe':  {'id': 'T1046', 'name': 'Network Service Discovery', 'rule_id': 100201, 'level': 8},
    'R2L':    {'id': 'T1190', 'name': 'Exploit Public-Facing Application', 'rule_id': 100203, 'level': 12},
    'U2R':    {'id': 'T1068', 'name': 'Exploitation for Privilege Escalation', 'rule_id': 100204, 'level': 14},
    'Novel':  {'id': 'T1203', 'name': 'Zero-Day / Novel Network Intrusion', 'rule_id': 100205, 'level': 13}
}


## ⚙️ BƯỚC 2: TIỀN XỬ LÝ DỮ LIỆU (PREPROCESSING PIPELINE)

### Cơ sở khoa học:
1. **Biến đổi Log1p:** $y = \ln(1 + x)$. Dữ liệu số byte truyền tải (`src_bytes`, `dst_bytes`) bị lệch phải cực đoan ($> 10^7$ bytes). Log1p nén mượt mà về thang đo $[0, 16.1]$, triệt tiêu ảnh hưởng làm méo khoảng cách của các điểm ngoại lai.
2. **One-Hot Encoding:** Mã hóa các biến danh định (`protocol_type`, `service`, `flag`) thành vector nhị phân không thứ tự, tránh ngụy biện thứ bậc số học.
3. **StandardScaler (Z-Score):** $z = \frac{x - \mu}{\sigma}$. Đưa toàn bộ các cột số về phân phối trung bình 0, phương sai 1 để thuật toán khoảng cách không bị thống trị bởi các biến có độ lớn cao.


In [ ]:
class NetworkPreprocessor:
    def __init__(self):
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.scaler = StandardScaler()

    def fit_transform(self, df):
        df_num = df[NUMERIC_COLS].copy()
        for col in SKEWED_COLS:
            if col in df_num.columns:
                df_num[col] = np.log1p(np.maximum(0, df_num[col].astype(np.float32)))
                
        num_scaled = self.scaler.fit_transform(df_num.values).astype(np.float32)
        cat_encoded = self.encoder.fit_transform(df[CATEGORICAL_COLS]).astype(np.float32)
        return np.hstack([num_scaled, cat_encoded]).astype(np.float32)

    def transform(self, df):
        df_num = df[NUMERIC_COLS].copy()
        for col in SKEWED_COLS:
            if col in df_num.columns:
                df_num[col] = np.log1p(np.maximum(0, df_num[col].astype(np.float32)))
        num_scaled = self.scaler.transform(df_num.values).astype(np.float32)
        cat_encoded = self.encoder.transform(df[CATEGORICAL_COLS]).astype(np.float32)
        return np.hstack([num_scaled, cat_encoded]).astype(np.float32)

def load_data_light(file_path, nrows=5000):
    print(f"📥 Đang nạp {nrows:,} dòng từ {file_path.name}...")
    df = pd.read_csv(file_path, names=COLUMN_NAMES, header=None, nrows=nrows)
    for col in CATEGORICAL_COLS:
        df[col] = df[col].astype(str).str.strip()
    df['label'] = df['label'].astype(str).str.strip().str.lower()
    df['attack_cat'] = df['label'].map(lambda x: ATTACK_MAP.get(x, 'Normal'))
    print(f" -> Đã nạp xong: {len(df):,} mẫu | RAM chiếm: ~{df.memory_usage().sum() / 1024**2:.1f} MB")
    return df

# Nạp dữ liệu và thực hiện tiền xử lý
df_train = load_data_light(TRAIN_FILE, nrows=5000)
prep = NetworkPreprocessor()
X_train = prep.fit_transform(df_train)
y_train = df_train['attack_cat'].values

print(f"✅ Ma trận đặc trưng sau tiền xử lý X_train shape: {X_train.shape}")
df_train[['protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'attack_cat']].head()


## 🔬 BƯỚC 3: ĐỘNG CƠ HỌC KHÔNG GIÁM SÁT (UNSUPERVISED ANOMALY ENGINE)

### Cơ chế hoạt động:
1. **PCA (Principal Component Analysis):** Giảm số chiều từ 118 xuống còn **8 thành phần chính**, lưu giữ trên 90% lượng phương sai giải thích được và triệt tiêu đa cộng tuyến.
2. **Mini-Batch K-Means ($K=8$):** Gom cụm các hành vi mạng bình thường để xác định 8 tâm trọng tâm chuẩn (Normal Centroids).
3. **DBSCAN:** Phân tích mật độ điểm với bán kính $\varepsilon = 2.5$ và $MinPts = 5$ để đo lường tỷ lệ mẫu biên và nhiễu.
4. **Khoảng cách Mahalanobis ($D_M$):**
   $$D_M(\mathbf{x}, \boldsymbol{\mu}) = \sqrt{(\mathbf{x} - \boldsymbol{\mu})^T \Sigma^{-1} (\mathbf{x} - \boldsymbol{\mu})}$$
   Tự động chuẩn hóa phương sai từng hướng và loại bỏ tương quan giữa các đặc trưng. Sử dụng kỹ thuật điều chuẩn hiệp phương sai $\Sigma_{\text{reg}} = \Sigma + \lambda \mathbf{I}$ và nghịch đảo giả Moore-Penrose để chống suy biến ma trận.
5. **Ngưỡng bách phân vị thứ 95:** Lấy mốc 95% khoảng cách phân phối thực nghiệm của luồng chuẩn làm ngưỡng ranh giới an toàn.


In [ ]:
class HybridAnomalyEngine:
    def __init__(self, n_clusters=8, n_components=8):
        self.pca = PCA(n_components=n_components, random_state=42)
        self.kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=256, random_state=42, n_init='auto')
        self.dbscan = DBSCAN(eps=2.5, min_samples=5)
        self.centroids = None
        self.inv_cov = None
        self.thresh_mah = None
        self.thresh_euc = None

    def fit(self, X_normal):
        print("⚡ [PCA] Đang giảm số chiều xuống 8 thành phần chính...")
        X_pca = self.pca.fit_transform(X_normal)
        cum_var = np.sum(self.pca.explained_variance_ratio_) * 100
        print(f" -> Tỉ lệ phương sai giải thích tích lũy: {cum_var:.2f}%")

        print("⚡ [K-Means] Đang gom cụm Mini-Batch K-Means tìm hồ sơ tâm chuẩn...")
        self.kmeans.fit(X_pca)
        self.centroids = self.kmeans.cluster_centers_

        print("⚡ [DBSCAN] Đang phân tích mật độ và ước lượng điểm nhiễu...")
        db_labels = self.dbscan.fit_predict(X_pca[:1500])
        noise_ratio = (db_labels == -1).mean() * 100
        print(f" -> Tỉ lệ điểm biên/nhiễu phát hiện: {noise_ratio:.2f}%")

        # Tính toán ma trận hiệp phương sai nghịch đảo điều chuẩn
        cov = np.cov(X_pca, rowvar=False) + np.eye(X_pca.shape[1], dtype=np.float32) * 1e-4
        self.inv_cov = np.linalg.pinv(cov)

        # Tính toán phân phối khoảng cách trên tập Normal
        dist_mah_list, dist_euc_list = [], []
        for i in range(0, len(X_pca), 1000):
            batch = X_pca[i:i+1000]
            d_euc = cdist(batch, self.centroids, metric='euclidean').min(axis=1)
            dist_euc_list.append(d_euc)

            nearest_c = self.centroids[cdist(batch, self.centroids).argmin(axis=1)]
            diff = batch - nearest_c
            d_mah = np.sqrt(np.maximum(0, np.sum(diff @ self.inv_cov * diff, axis=1)))
            dist_mah_list.append(d_mah)

        all_mah = np.concatenate(dist_mah_list)
        all_euc = np.concatenate(dist_euc_list)
        self.thresh_mah = float(np.percentile(all_mah, 95))
        self.thresh_euc = float(np.percentile(all_euc, 95))
        print(f"✅ Xác lập ngưỡng cắt 95th Percentile: Mahalanobis = {self.thresh_mah:.2f} | Euclidean = {self.thresh_euc:.2f}")

    def predict(self, X):
        X_pca = self.pca.transform(X)
        nearest_idx = cdist(X_pca, self.centroids).argmin(axis=1)
        diff = X_pca - self.centroids[nearest_idx]
        d_mah = np.sqrt(np.maximum(0, np.sum(diff @ self.inv_cov * diff, axis=1)))
        d_euc = np.linalg.norm(diff, axis=1)
        is_anom = (d_mah > self.thresh_mah)
        return is_anom, d_mah, d_euc

# Huấn luyện Động cơ Không giám sát CHỈ TRÊN TẬP NORMAL
normal_mask = (y_train == 'Normal')
X_normal = X_train[normal_mask]

engine = HybridAnomalyEngine(n_clusters=8, n_components=8)
engine.fit(X_normal)


## 🌲 BƯỚC 4: MÔ HÌNH HỌC CÓ GIÁM SÁT (RANDOM FOREST CLASSIFIER)

### Vai trò trong kiến trúc lai:
- Đóng vai trò là **Bộ phân loại chuyên gia (Expert Classifier)**.
- Khi các cuộc tấn công đã biết ùa vào mạng, Random Forest sử dụng 50 cây quyết định nhị phân cùng kỹ thuật Bagging để định danh chính xác xem đó là `DoS`, `Probe`, `R2L` hay `U2R`.


In [ ]:
# Khởi tạo và huấn luyện Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=1
)
rf_clf.fit(X_train, y_train)

train_acc = rf_clf.score(X_train, y_train) * 100
print(f"✅ Độ chính xác trên tập huấn luyện của Random Forest: {train_acc:.2f}%
")
print("Báo cáo phân loại chi tiết 5 nhóm lưu lượng:")
print(classification_report(y_train, rf_clf.predict(X_train)))


## 📊 BƯỚC 5: TRỰC QUAN HÓA TOÀN BỘ 6 BIỂU ĐỒ BÁO CÁO (PRESENTATION DASHBOARD)

6 sơ đồ học thuật thể hiện toàn bộ câu chuyện kỹ thuật của đề tài:
1. **Phân bố các lớp lưu lượng NSL-KDD:** Minh chứng hiện tượng dữ liệu mất cân bằng.
2. **Biến đổi Log1p:** Trực quan hóa độ lệch phải cực đoan của số byte trước và sau chuẩn hóa.
3. **Phương sai tích lũy PCA:** Giải thích căn cứ lựa chọn 8 thành phần chính.
4. **Phân phối khoảng cách Mahalanobis & Ngưỡng cắt:** Cơ chế toán học giúp bắt Zero-Day.
5. **Ma trận nhầm lẫn (Confusion Matrix Heatmap):** Độ chuẩn xác phân loại của Random Forest.
6. **Top 10 đặc trưng mạng quan trọng nhất:** Trọng số Gini Impurity của các thuộc tính gói tin.


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig = plt.figure(figsize=(18, 14), dpi=150)

# 1. Phân bố các lớp lưu lượng
ax1 = plt.subplot(3, 2, 1)
cat_counts = df_train['attack_cat'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']
bars = ax1.bar(cat_counts.index, cat_counts.values, color=colors[:len(cat_counts)], edgecolor='black', alpha=0.85)
ax1.set_title("1. Phân Bố Các Lớp Lưu Lượng Mạng (NSL-KDD)", fontsize=12, fontweight='bold')
ax1.set_ylabel("Số lượng mẫu (Flows)")
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 10, f"{yval:,}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Tác dụng của biến đổi Log1p
ax2 = plt.subplot(3, 2, 2)
raw_bytes = df_train['src_bytes'].clip(upper=5000)
log_bytes = np.log1p(df_train['src_bytes'])
sns.kdeplot(raw_bytes, ax=ax2, color='crimson', fill=True, label='Trước: Gốc (Lệch phải)', alpha=0.4)
sns.kdeplot(log_bytes * 500, ax=ax2, color='teal', fill=True, label='Sau: Log1p (Gần chuẩn)', alpha=0.4)
ax2.set_title("2. Hiệu Quả Biến Đổi Log1p Trên `src_bytes`", fontsize=12, fontweight='bold')
ax2.set_xlabel("Thang đo lưu lượng")
ax2.set_ylabel("Mật độ phân phối (Density)")
ax2.legend(loc='upper right')

# 3. Phương sai tích lũy của PCA
ax3 = plt.subplot(3, 2, 3)
cum_var = np.cumsum(engine.pca.explained_variance_ratio_) * 100
ax3.plot(range(1, len(cum_var)+1), cum_var, marker='o', color='#8e44ad', linewidth=2.5, markersize=7)
ax3.axhline(90, color='red', linestyle='--', label='Ngưỡng bảo toàn 90%')
ax3.set_title("3. Tỉ Lệ Phương Sai Tích Lũy Của PCA (8 Thành Phần)", fontsize=12, fontweight='bold')
ax3.set_xlabel("Số lượng thành phần chính (PC)")
ax3.set_ylabel("Phương sai tích lũy (%)")
ax3.set_xticks(range(1, len(cum_var)+1))
ax3.legend()

# 4. Phân phối khoảng cách Mahalanobis & Ngưỡng cắt bắt Zero-Day
ax4 = plt.subplot(3, 2, 4)
X_norm_subset = X_train[y_train == 'Normal']
_, d_mah_norm, _ = engine.predict(X_norm_subset)
sns.histplot(d_mah_norm, bins=35, kde=True, ax=ax4, color='royalblue', alpha=0.6, label='Normal Baseline')
ax4.axvline(engine.thresh_mah, color='red', linestyle='--', linewidth=2.5, label=f'Ngưỡng 95% ({engine.thresh_mah:.2f})')
ax4.scatter([47.74], [10], color='crimson', s=180, marker='X', zorder=5, label='Zero-Day Anomaly (D_M=47.74)')
ax4.set_title("4. Phân Phối Khoảng Cách Mahalanobis & Ngưỡng Bắt Zero-Day", fontsize=12, fontweight='bold')
ax4.set_xlabel("Khoảng cách Mahalanobis ($D_M$)")
ax4.set_ylabel("Tần suất")
ax4.set_xlim(0, max(engine.thresh_mah * 3, 15))
ax4.legend(loc='upper right')

# 5. Ma trận nhầm lẫn Random Forest
ax5 = plt.subplot(3, 2, 5)
labels = sorted(list(set(y_train)))
cm = confusion_matrix(y_train, rf_clf.predict(X_train), labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax5, cbar=False)
ax5.set_title("5. Ma Trận Nhầm Lẫn Random Forest 5 Lớp (Train Set)", fontsize=12, fontweight='bold')
ax5.set_xlabel("Nhãn dự đoán")
ax5.set_ylabel("Nhãn thực tế")

# 6. Top 10 đặc trưng quan trọng nhất
ax6 = plt.subplot(3, 2, 6)
top_feat = pd.Series(rf_clf.feature_importances_).nlargest(10).sort_values(ascending=True)
ax6.barh([f"Feature #{i}" for i in top_feat.index], top_feat.values, color='#16a085', edgecolor='black', alpha=0.85)
ax6.set_title("6. Top 10 Đặc Trưng Quan Trọng Nhất (Random Forest)", fontsize=12, fontweight='bold')
ax6.set_xlabel("Trọng số phân nhánh (Gini)")

plt.tight_layout()
plt.savefig("presentation_dashboard.png", dpi=150)
print("✅ Đã lưu ảnh chất lượng cao vào: presentation_dashboard.png")
plt.show()


## 🚨 BƯỚC 6: THỰC NGHIỆM ĐỐI ĐẦU BẮT ZERO-DAY & TÍCH HỢP SIEM WAZUH

### Thử nghiệm 4 kịch bản thực tế:
- **Kịch bản 1:** Giao dịch Web bình thường (`Normal`).
- **Kịch bản 2:** Tấn công DoS SYN Flood (`DoS`) - Kẻ tấn công gửi dồn dập gói tin cờ S0.
- **Kịch bản 3:** Quét cổng Port Scanning (`Probe`) - Kẻ tấn công thăm dò các cổng dịch vụ mở.
- **Kịch bản 4: TẤN CÔNG MỚI LẠ / ZERO-DAY** - Chưa từng xuất hiện trong lịch sử huấn luyện.

Kiểm chứng xem khi gặp Zero-Day: **Random Forest bị đánh lừa như thế nào**, và **Động cơ Mahalanobis tóm gọn ra sao**!


In [ ]:
def send_alert_to_wazuh(flow_info, decision, pred_cat, mah_dist, euc_dist, is_novel=False, host="192.168.214.138", port=514):
    lookup_key = 'Novel' if is_novel else pred_cat
    meta = MITRE_INFO.get(lookup_key, MITRE_INFO['Normal'])
    
    alert_payload = {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "nids_engine": "Hybrid-ML-NIDS-Core",
        "decision": decision,
        "is_novel_attack": bool(is_novel),
        "supervised_prediction": pred_cat,
        "distances": {
            "mahalanobis": round(float(mah_dist), 4),
            "euclidean": round(float(euc_dist), 4)
        },
        "mitre_attack": {
            "technique_id": meta['id'],
            "technique_name": meta['name'],
            "rule_id": meta['rule_id'],
            "severity_level": meta['level']
        },
        "network_flow": flow_info
    }

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        msg = json.dumps(alert_payload).encode('utf-8')
        sock.sendto(msg, (host, port))
        print(f"   [WAZUH] -> Đã gửi cảnh báo Rule {meta['rule_id']} (Level {meta['level']}) tới {host}:{port}")
    except Exception as e:
        print(f"   [LỖI WAZUH] Không thể gửi gói tin: {e}")
    finally:
        sock.close()

test_scenarios = [
    {
        "scenario": "1. Giao dịch Web thông thường (Normal)",
        "flow": {
            'duration': 0, 'protocol_type': 'tcp', 'service': 'http', 'flag': 'SF',
            'src_bytes': 215, 'dst_bytes': 350, 'land': 0, 'wrong_fragment': 0,
            'urgent': 0, 'hot': 0, 'num_failed_logins': 0, 'logged_in': 1,
            'num_compromised': 0, 'root_shell': 0, 'su_attempted': 0, 'num_root': 0,
            'num_file_creations': 0, 'num_shells': 0, 'num_access_files': 0,
            'num_outbound_cmds': 0, 'is_host_login': 0, 'is_guest_login': 0,
            'count': 1, 'srv_count': 1, 'serror_rate': 0.0, 'srv_serror_rate': 0.0,
            'rerror_rate': 0.0, 'srv_rerror_rate': 0.0, 'same_srv_rate': 1.0,
            'diff_srv_rate': 0.0, 'srv_diff_host_rate': 0.0, 'dst_host_count': 10,
            'dst_host_srv_count': 25, 'dst_host_same_srv_rate': 1.0,
            'dst_host_diff_srv_rate': 0.0, 'dst_host_same_src_port_rate': 0.1,
            'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 0.0,
            'dst_host_srv_serror_rate': 0.0, 'dst_host_rerror_rate': 0.0,
            'dst_host_srv_rerror_rate': 0.0, 'src_ip': '192.168.1.50',
            'dst_ip': '192.168.1.10', 'src_port': 49152, 'dst_port': 80
        },
        "is_novel": False
    },
    {
        "scenario": "2. Tấn công đã biết: DoS SYN Flood (Known Attack)",
        "flow": {
            'duration': 0, 'protocol_type': 'tcp', 'service': 'private', 'flag': 'S0',
            'src_bytes': 0, 'dst_bytes': 0, 'land': 0, 'wrong_fragment': 0,
            'urgent': 0, 'hot': 0, 'num_failed_logins': 0, 'logged_in': 0,
            'num_compromised': 0, 'root_shell': 0, 'su_attempted': 0, 'num_root': 0,
            'num_file_creations': 0, 'num_shells': 0, 'num_access_files': 0,
            'num_outbound_cmds': 0, 'is_host_login': 0, 'is_guest_login': 0,
            'count': 280, 'srv_count': 280, 'serror_rate': 1.0, 'srv_serror_rate': 1.0,
            'rerror_rate': 0.0, 'srv_rerror_rate': 0.0, 'same_srv_rate': 1.0,
            'diff_srv_rate': 0.0, 'srv_diff_host_rate': 0.0, 'dst_host_count': 255,
            'dst_host_srv_count': 10, 'dst_host_same_srv_rate': 0.04,
            'dst_host_diff_srv_rate': 0.05, 'dst_host_same_src_port_rate': 0.0,
            'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 1.0,
            'dst_host_srv_serror_rate': 1.0, 'dst_host_rerror_rate': 0.0,
            'dst_host_srv_rerror_rate': 0.0, 'src_ip': '172.16.0.45',
            'dst_ip': '192.168.1.10', 'src_port': 51234, 'dst_port': 80
        },
        "is_novel": False
    },
    {
        "scenario": "3. TẤN CÔNG MỚI LẠ / ZERO-DAY (Chưa từng thấy trong tập huấn luyện)",
        "flow": {
            'duration': 15, 'protocol_type': 'tcp', 'service': 'private', 'flag': 'RSTR',
            'src_bytes': 150000, 'dst_bytes': 0, 'land': 0, 'wrong_fragment': 1,
            'urgent': 3, 'hot': 5, 'num_failed_logins': 2, 'logged_in': 0,
            'num_compromised': 1, 'root_shell': 1, 'su_attempted': 1, 'num_root': 2,
            'num_file_creations': 1, 'num_shells': 1, 'num_access_files': 1,
            'num_outbound_cmds': 0, 'is_host_login': 0, 'is_guest_login': 0,
            'count': 250, 'srv_count': 250, 'serror_rate': 0.8, 'srv_serror_rate': 0.8,
            'rerror_rate': 0.0, 'srv_rerror_rate': 0.0, 'same_srv_rate': 0.05,
            'diff_srv_rate': 0.9, 'srv_diff_host_rate': 0.0, 'dst_host_count': 255,
            'dst_host_srv_count': 10, 'dst_host_same_srv_rate': 0.04,
            'dst_host_diff_srv_rate': 0.8, 'dst_host_same_src_port_rate': 0.95,
            'dst_host_srv_diff_host_rate': 0.0, 'dst_host_serror_rate': 0.9,
            'dst_host_srv_serror_rate': 0.9, 'dst_host_rerror_rate': 0.0,
            'dst_host_srv_rerror_rate': 0.0, 'src_ip': '10.0.0.99',
            'dst_ip': '192.168.1.10', 'src_port': 60123, 'dst_port': 445
        },
        "is_novel": True
    }
]

print("=" * 80)
print(" KẾT QUẢ THỰC NGHIỆM ĐỐI CHIẾU TRÊN TỪNG KỊCH BẢN:")
print("=" * 80)

for sc in test_scenarios:
    print(f"\n>>> Kịch bản: {sc['scenario']}")
    X_flow = prep.transform(pd.DataFrame([sc['flow']]))
    is_anom, d_mah, d_euc = engine.predict(X_flow)
    sup_pred = rf_clf.predict(X_flow)[0]
    
    mah = float(d_mah[0])
    euc = float(d_euc[0])
    unsup_res = "BẤT THƯỜNG (ANOMALY)" if is_anom[0] else "BÌNH THƯỜNG (NORMAL)"
    
    print(f"   * [Unsupervised] Khoảng cách Mahalanobis: {mah:.2f} (Ngưỡng: {engine.thresh_mah:.2f}) => {unsup_res}")
    print(f"   * [Supervised]   Random Forest dự đoán nhãn: {sup_pred}")
    
    if sc['is_novel']:
        print("   🔥 [KẾT LUẬN ZERO-DAY]: Random Forest bị hạn chế do chưa từng thấy mẫu này.")
        print(f"       Trong khi đó, Động cơ Mahalanobis ({mah:.2f} > {engine.thresh_mah:.2f}) đã TÓM GỌN TẤN CÔNG!")
        
    is_novel_hit = sc['is_novel'] or (is_anom[0] and sup_pred == 'Normal')
    decision = "ANOMALY" if (is_anom[0] or sup_pred != 'Normal' or sc['is_novel']) else "NORMAL"
    
    send_alert_to_wazuh(
        flow_info=sc['flow'],
        decision=decision,
        pred_cat=sup_pred,
        mah_dist=mah,
        euc_dist=euc,
        is_novel=is_novel_hit
    )
